<a href="https://colab.research.google.com/github/Olmioris/LLM-instruction-tuning-qwen2-qlora/blob/main/%D0%9E%D1%86%D0%B5%D0%BD%D0%BA%D0%B0_%D0%BD%D0%B0_Qwen2_0_5B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Проверяем, что веса модели есть
!ls -lh /content/drive/MyDrive/Qwen2-0.5B-SFT-MultiDomain

# Лёгкая модель
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
OUTPUT_DIR = "/content/drive/MyDrive/Qwen2-0.5B-SFT-MultiDomain"

!pip install -q transformers peft accelerate lm_eval

from lm_eval import evaluator
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

Mounted at /content/drive
total 16M
-rw------- 1 root root 1.1K Aug  6 21:38 adapter_config.json
-rw------- 1 root root 4.2M Aug  6 21:38 adapter_model.safetensors
-rw------- 1 root root  328 Aug  6 21:38 chat_template.jinja
drwx------ 2 root root 4.0K Aug  6 19:07 checkpoint-80
-rw------- 1 root root 1.8K Aug  6 21:38 config.json
-rw------- 1 root root 1.5K Aug  6 21:39 README.md
-rw------- 1 root root  447 Aug  6 21:38 tokenizer_config.json
-rw------- 1 root root  11M Aug  6 21:38 tokenizer.json
-rw------- 1 root root 5.6K Aug  6 21:38 training_args.bin
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.2 MB/s eta 0:00:0

In [ ]:
results_before = evaluator.simple_evaluate(
    model="hf",
    model_args=f"pretrained={MODEL_NAME},dtype=float32",
    tasks=["hellaswag"],
    num_fewshot=0,
    limit=500,
    batch_size=1,
)

hella_before = results_before["results"]["hellaswag"]
acc_norm_key_before = next(k for k in hella_before.keys() if "acc_norm" in k)
print("Baseline Hellaswag acc_norm:", hella_before[acc_norm_key_before])

        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/10042 [00:00<?, ? examples/s]

Map:   0%|          | 0/39905 [00:00<?, ? examples/s]

Map:   0%|          | 0/10042 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 2000/2000 [01:37<00:00, 20.51it/s]


Baseline Hellaswag acc_norm: 0.486


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
results_after = evaluator.simple_evaluate(
    model="hf",
    model_args=f"pretrained={MODEL_NAME},peft={OUTPUT_DIR},dtype=float32",
    tasks=["hellaswag"],
    num_fewshot=0,
    limit=500,
    batch_size=1,
)

hella_after = results_after["results"]["hellaswag"]
acc_norm_key_after = next(k for k in hella_after.keys() if "acc_norm" in k)
print("Finetuned Hellaswag acc_norm:", hella_after[acc_norm_key_after])

        instruct or chat variant but chat template is not applied. Recommend setting `apply_chat_template` (optionally
        `fewshot_as_multiturn`).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 2000/2000 [02:15<00:00, 14.78it/s]


Finetuned Hellaswag acc_norm: 0.484


In [ ]:
baseline_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="cpu",
)

finetuned_model = PeftModel.from_pretrained(
    baseline_model,
    OUTPUT_DIR,
)

finetuned_tokenizer = baseline_tokenizer
finetuned_model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

In [ ]:
def generate_baseline(prompt, max_new_tokens=200):
    inputs = baseline_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = baseline_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return baseline_tokenizer.decode(out[0], skip_special_tokens=True)

def generate_finetuned(prompt, max_new_tokens=200):
    inputs = finetuned_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = finetuned_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return finetuned_tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "Explain the difference between supervised and reinforcement learning.",
    "Give three recommendations for improving customer support in an online bank.",
    "Describe potential risks of using large language models in cybersecurity.",
]

for i, p in enumerate(test_prompts, 1):
    print(f"\n=== PROMPT {i} ===\n{p}\n")
    print("BASELINE:\n", generate_baseline(p))
    print("\nFINETUNED:\n", generate_finetuned(p))


=== PROMPT 1 ===
Explain the difference between supervised and reinforcement learning.

BASELINE:
 Explain the difference between supervised and reinforcement learning. Supervised Learning: The goal of supervised learning is to learn a model that can be used to predict outputs based on inputs. This type of learning is often done through training a machine with labeled data, where the output is predicted by the model given its input. In this context, we are using labeled data for training our model.

Reinforcement Learning: Reinforcement learning (RL) is an approach to learning that involves interacting with the environment to obtain rewards or penalties as you make decisions. RL is used in a variety of applications such as game playing, robotics, and autonomous vehicles. RL differs from supervised learning in that it does not require labeled data for training. Instead, RL uses feedback from the environment to update the model's policy or value function. This process is known as "rewar